# Training our own car counter from scratch (Prototype 3)
**Group 8.** We train a small detector that has never seen aerial images (YOLO11n, trained on everyday photos) on 515 cars marked by hand in COWC. Then we test it on a tile it never saw, against the ready-made DOTA model.
*Use a GPU: Runtime > Change runtime type > T4 GPU. Training takes ~5 min on GPU (it took ~1 hour on our 2-core CPU).*

In [ ]:
!pip -q install ultralytics

In [ ]:
"""Car counting from overhead images with an open, pretrained detector.

Detector: Ultralytics YOLO11 oriented-box model pretrained on DOTA v1
(aerial images, class "small vehicle" = cars). We do NOT train anything:
the point is to test whether an off-the-shelf open model is enough.
"""
import numpy as np
from PIL import Image
from scipy import ndimage
from scipy.optimize import linear_sum_assignment

Image.MAX_IMAGE_PIXELS = None


def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def car_class_ids(model):
    """'small vehicle' in the DOTA model; 'car' in our own COWC-trained model."""
    return [i for i, n in model.names.items() if n.replace("-", " ").lower() in ("small vehicle", "car")]


def enhance(im):
    """Stretch contrast and sharpen: helps on hazy, compressed aerial photos."""
    from PIL import ImageOps, ImageFilter
    im = ImageOps.autocontrast(im, cutoff=1)
    return im.filter(ImageFilter.UnsharpMask(radius=3, percent=150, threshold=2))


def detect_cars(model, img, scale=1.0, tile=1024, overlap=160, conf=0.25, imgsz=1024, dedup_px=10, sharpen=False):
    """Run the detector on a large image by tiling it.

    scale: resize factor applied before detection (to match the detector's preferred car size).
    dedup_px: two detections closer than this (original pixels) are the same car.
    Returns an array (N, 4): x, y, confidence, box length (all in ORIGINAL image pixels).
    """
    h0, w0 = img.shape[:2]
    pil = Image.fromarray(img)
    if scale != 1.0:
        pil = pil.resize((int(w0 * scale), int(h0 * scale)), Image.LANCZOS)
    if sharpen:
        pil = enhance(pil)
    im = np.array(pil)
    h, w = im.shape[:2]
    step = tile - overlap
    cls = car_class_ids(model)
    out = []
    ys = list(range(0, max(h - overlap, 1), step))
    xs = list(range(0, max(w - overlap, 1), step))
    for y in ys:
        for x in xs:
            crop = im[y:y + tile, x:x + tile]
            if crop.shape[0] < 32 or crop.shape[1] < 32:
                continue
            r = model.predict(crop, imgsz=imgsz, conf=conf, classes=cls, verbose=False)[0]
            if r.obb is not None:                      # oriented boxes (DOTA model)
                if len(r.obb) == 0:
                    continue
                xywh = r.obb.xywhr.cpu().numpy()[:, :4]; cf = r.obb.conf.cpu().numpy()
            else:                                      # ordinary boxes (our own model)
                if r.boxes is None or len(r.boxes) == 0:
                    continue
                xywh = r.boxes.xywh.cpu().numpy(); cf = r.boxes.conf.cpu().numpy()
            for (cx, cy, bw, bh), c in zip(xywh, cf):
                out.append((x + cx, y + cy, c, max(bw, bh)))
    if not out:
        return np.zeros((0, 4))
    d = np.array(out, dtype=float)
    # merge duplicates from overlapping tiles: keep highest confidence within dedup_px
    dedup_px = dedup_px * scale          # given in original-image pixels
    from scipy.spatial import cKDTree
    order = np.argsort(-d[:, 2])
    tree = cKDTree(d[:, :2])
    removed = np.zeros(len(d), bool)
    keep = []
    for i in order:                       # greedy: keep the most confident, drop its neighbours
        if removed[i]:
            continue
        keep.append(i)
        for j in tree.query_ball_point(d[i, :2], dedup_px):
            removed[j] = True
    d = d[keep]
    d[:, :2] /= scale
    d[:, 3] /= scale
    return d


def gt_points(annot_path):
    """COWC ground truth: one small mark at the centre of every car."""
    a = np.array(Image.open(annot_path))
    if a.ndim == 3:
        a = a[..., :3].max(axis=2) if a.shape[2] >= 3 else a[..., 0]
    lab, n = ndimage.label(a > 0)
    if n == 0:
        return np.zeros((0, 2))
    c = ndimage.center_of_mass(np.ones_like(lab), lab, range(1, n + 1))
    return np.array([(x, y) for y, x in c])


def match(det_xy, gt_xy, radius):
    """One-to-one matching of detections to true cars within `radius` pixels."""
    if len(det_xy) == 0 or len(gt_xy) == 0:
        return 0, len(det_xy), len(gt_xy)
    D = np.hypot(det_xy[:, None, 0] - gt_xy[None, :, 0], det_xy[:, None, 1] - gt_xy[None, :, 1])
    cost = np.where(D <= radius, D, 1e6)
    r, c = linear_sum_assignment(cost)
    tp = int((cost[r, c] < 1e6).sum())
    return tp, len(det_xy) - tp, len(gt_xy) - tp


def draw(img, det_xy, gt_xy=None, matched=None, r=9, path=None, max_side=2400):
    """Save a picture with detections (blue rings) and, if given, true cars (orange dots)."""
    from PIL import ImageDraw
    im = Image.fromarray(img).copy()
    dr = ImageDraw.Draw(im)
    if gt_xy is not None:
        for x, y in gt_xy:
            dr.ellipse([x - 3, y - 3, x + 3, y + 3], fill=(235, 104, 52))
    for x, y in det_xy:
        dr.ellipse([x - r, y - r, x + r, y + r], outline=(42, 120, 214), width=3)
    if max(im.size) > max_side:
        f = max_side / max(im.size)
        im = im.resize((int(im.size[0] * f), int(im.size[1] * f)), Image.LANCZOS)
    if path:
        im.save(path, quality=88)
    return im


In [ ]:
"""Turn COWC tiles (photo + one dot per car) into a YOLO training set.

- Every image is resampled to TARGET_GSD (0.10 m/pixel), so a car is ~44 px long
  whatever the source. At prediction time we resample IGN photos the same way.
- Each car dot becomes a square box BOX_M metres wide (COWC gives no car outline).
- For every tile we also make a "soft" copy: shrunk to 0.25 m/pixel, JPEG-compressed
  and enlarged back. It mimics the softer IGN photos so the model learns both looks.
Usage: python train_prep.py <out_dir> <tile.png> [<tile.png> ...]   (labels: <tile>_Annotated_Cars.png)
"""
import sys, os, io
import numpy as np
from PIL import Image

Image.MAX_IMAGE_PIXELS = None
SRC_GSD = 300 / 2220        # COWC Potsdam tiles: 300 m across in 2220 px
TARGET_GSD = 0.10
BOX_M = 3.6
CROP = 640
STEP = 560


def soften(im, gsd_from, gsd_soft=0.25, quality=70):
    w, h = im.size
    small = im.resize((int(w * gsd_from / gsd_soft), int(h * gsd_from / gsd_soft)), Image.LANCZOS)
    buf = io.BytesIO(); small.save(buf, "JPEG", quality=quality); buf.seek(0)
    return Image.open(buf).convert("RGB").resize((w, h), Image.BICUBIC)


def write_crops(im, pts, out, split, stem):
    os.makedirs(f"{out}/images/{split}", exist_ok=True); os.makedirs(f"{out}/labels/{split}", exist_ok=True)
    a = np.array(im); H, W = a.shape[:2]; b = BOX_M / TARGET_GSD; n = 0
    for y in range(0, max(H - CROP, 0) + 1, STEP):
        for x in range(0, max(W - CROP, 0) + 1, STEP):
            sel = pts[(pts[:, 0] >= x) & (pts[:, 0] < x + CROP) & (pts[:, 1] >= y) & (pts[:, 1] < y + CROP)]
            name = f"{stem}_{x}_{y}"
            Image.fromarray(a[y:y + CROP, x:x + CROP]).save(f"{out}/images/{split}/{name}.jpg", quality=92)
            with open(f"{out}/labels/{split}/{name}.txt", "w") as f:
                for px, py in sel:
                    cx, cy = (px - x) / CROP, (py - y) / CROP
                    f.write(f"0 {cx:.5f} {cy:.5f} {b / CROP:.5f} {b / CROP:.5f}\n")
            n += 1
    return n




## 1. Download COWC tiles (4 for training, 1 for validation, 1 for the final test)

In [ ]:
import os, urllib.request
BASE = "https://gdo152.llnl.gov/cowc/download/cowc/datasets/ground_truth_sets/Potsdam_ISPRS/"
TILES = ["2_10", "2_11", "2_13", "3_10", "6_8", "2_12"]   # 6_8 = validation, 2_12 = test
os.makedirs("cowc", exist_ok=True)
for t in TILES:
    for suffix in ("_RGB.png", "_RGB_Annotated_Cars.png"):
        f = f"cowc/top_potsdam_{t}{suffix}"
        if not os.path.exists(f):
            urllib.request.urlretrieve(BASE + f"top_potsdam_{t}{suffix}", f)
print({t: len(gt_points(f"cowc/top_potsdam_{t}_RGB_Annotated_Cars.png")) for t in TILES})

## 2. Build the training set (dots → 3.6 m boxes, 10 cm per pixel, plus blurred copies)

In [ ]:
import glob
out = "trainset"
for t in ["2_10", "2_11", "2_13", "3_10", "6_8"]:
    path = f"cowc/top_potsdam_{t}_RGB.png"
    split = "val" if t == "6_8" else "train"
    im = Image.open(path).convert("RGB"); pts = gt_points(path.replace(".png", "_Annotated_Cars.png"))
    f = SRC_GSD / TARGET_GSD
    big = im.resize((int(im.width * f), int(im.height * f)), Image.LANCZOS)
    stem = os.path.basename(path)[:-4]
    write_crops(big, pts * f, out, split, stem)
    write_crops(soften(big, TARGET_GSD), pts * f, out, split, stem + "_soft")
open(f"{out}/data.yaml", "w").write(f"path: {os.path.abspath(out)}\ntrain: images/train\nval: images/val\nnames:\n  0: car\n")
print(len(glob.glob(out + "/images/train/*")), "training crops,", len(glob.glob(out + "/images/val/*")), "validation crops")

## 3. Train

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")   # trained on everyday photos only (COCO)
model.train(data="trainset/data.yaml", epochs=30, imgsz=640, batch=16, degrees=180, flipud=0.5, fliplr=0.5,
            mosaic=1.0, scale=0.3, hsv_v=0.3, project="runs", name="cowc_n", exist_ok=True, seed=0)
own = YOLO(str(model.trainer.best))

## 4. Choose the confidence cut-off on the validation tile only, then test on 2_12

In [ ]:
COWC_GSD = 300 / 2220; R = 1.5
img = load_rgb("cowc/top_potsdam_6_8_RGB.png"); gt = gt_points("cowc/top_potsdam_6_8_RGB_Annotated_Cars.png")
d = detect_cars(own, img, scale=COWC_GSD / 0.10, conf=0.05, dedup_px=R / COWC_GSD)
scores = {}
for c in np.arange(0.1, 0.75, 0.05):
    tp, fp, fn = match(d[d[:, 2] >= c][:, :2], gt, R / COWC_GSD); scores[round(c, 2)] = 2 * tp / (2 * tp + fp + fn)
CONF = max(scores, key=scores.get); print("chosen cut-off:", CONF)

img = load_rgb("cowc/top_potsdam_2_12_RGB.png"); gt = gt_points("cowc/top_potsdam_2_12_RGB_Annotated_Cars.png")
dota = YOLO("yolo11m-obb.pt")
for name, m, kw in [("ours", own, dict(scale=COWC_GSD / 0.10, conf=CONF)), ("ready-made", dota, dict(scale=1.5, conf=0.25))]:
    det = detect_cars(m, img, dedup_px=R / COWC_GSD, **kw); tp, fp, fn = match(det[:, :2], gt, R / COWC_GSD)
    print(f"{name:10s}: counted {len(det)}, right {tp}, false alarms {fp}, missed {fn} "
          f"-> found {tp/(tp+fn):.0%}, right {tp/(tp+fp):.0%}")

**Our result (CPU run, seed 0):** ours 110 counted, 73 right, 37 false alarms, 40 missed (65% found, 66% right); ready-made 98 counted, 92 right, 6 false alarms, 21 missed (81% found, 94% right). A GPU run may differ by a few cars.